In [4]:
import os, sys, pickle, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    _HERE = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _HERE = os.path.abspath('')

_ROOT     = os.path.normpath(os.path.join(_HERE, *(['..'] * 7)))
_SFERA_DB = os.path.join(_ROOT, 'sfera-db')
if os.path.isdir(_SFERA_DB) and _SFERA_DB not in sys.path:
    sys.path.insert(0, _SFERA_DB)
import sfera_db

WEIGHTS_DIR     = os.path.join(_HERE, 'weights')
CACHE_PATH      = os.path.join(WEIGHTS_DIR, 'pred_cache.pkl')
MANIFEST_PATH   = os.path.join(WEIGHTS_DIR, 'manifest.json')
ENTRY_THRESHOLD = 1.015  # signal when max(pred ratio) >= 1.5% gain

with open(CACHE_PATH, 'rb') as f:
    pred_df = pickle.load(f)
pred_df.index = pd.DatetimeIndex(pred_df.index)
TARGET_COLS = [c for c in pred_df.columns if c.startswith('d')]
n_days = len(TARGET_COLS)

# pred_df contains price ratios: predicted_close[t+k] / anchor  (~1.0, same as v2)
# Convert to % for display: (ratio - 1) * 100
pred_pct_df = (pred_df - 1) * 100

cactr = (sfera_db.query(
    "SELECT trade_date AS date, close_price AS close "
    "FROM bbgidx.index_total_return WHERE ticker='CACT' ORDER BY trade_date")
    .assign(date=lambda d: pd.to_datetime(d['date']))
    .set_index('date'))

common      = pred_df.index.intersection(cactr.index)
pred_df     = pred_df.loc[common]
pred_pct_df = pred_pct_df.loc[common]
close       = cactr.loc[common, 'close']
all_dates   = close.index

max_ratio  = pred_df.max(axis=1)
pct_signal = (max_ratio >= ENTRY_THRESHOLD).mean()

print(f'pred_cache: {len(pred_df):,} rows x {n_days} cols  {pred_df.index[0].date()} -> {pred_df.index[-1].date()}')
print(f'd1  ratio mean={pred_df["d1"].mean():.5f}  std={pred_df["d1"].std():.5f}')
print(f'd10 ratio mean={pred_df["d10"].mean():.5f}  std={pred_df["d10"].std():.5f}')
print(f'd1  pct   mean={pred_pct_df["d1"].mean():.3f}%  std={pred_pct_df["d1"].std():.3f}%')
print(f'signal ON (max ratio >= {ENTRY_THRESHOLD}): {pct_signal:.1%} of days')


pred_cache: 2,873 rows x 10 cols  2015-01-02 -> 2026-03-24
d1  ratio mean=1.00341  std=0.00608
d10 ratio mean=1.00104  std=0.04713
d1  pct   mean=0.341%  std=0.608%
signal ON (max ratio >= 1.015): 59.0% of days


In [5]:
# --- Variance check ---
print('Per-column std of price ratios (should be ~0.005-0.02, i.e. 0.5-2% spread):')
print(pred_df.std().round(5).to_string())
print()
print('Sample predictions (5 random rows) — % change from anchor:')
sample = pred_pct_df.sample(5, random_state=42).sort_index()
print(sample.round(3).to_string())


Per-column std of price ratios (should be ~0.005-0.02, i.e. 0.5-2% spread):
d1     0.00608
d2     0.01185
d3     0.01897
d4     0.02602
d5     0.03169
d6     0.03645
d7     0.04009
d8     0.04315
d9     0.04521
d10    0.04713

Sample predictions (5 random rows) — % change from anchor:
               d1     d2     d3     d4     d5     d6     d7     d8     d9    d10
2017-11-06 -0.577 -1.000 -0.936 -0.837 -2.534 -1.944 -2.486 -3.678 -4.609 -5.564
2018-08-31 -0.196 -0.406 -1.773 -2.806 -4.953 -7.768 -8.696 -4.050 -4.626 -5.476
2020-01-20  0.803  0.843  1.522 -0.319 -0.112 -0.008 -0.607 -0.582 -1.178 -0.877
2021-10-20 -0.157 -0.340 -1.018 -2.551 -2.621 -2.390 -2.545 -2.381 -1.117  0.369
2023-01-30  0.780  0.214  0.083 -0.192  0.597 -0.282 -0.129  0.666  1.552  1.993


In [6]:
# --- Load cycle metadata ---
cycles = []
if os.path.isfile(MANIFEST_PATH):
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    for key, info in sorted(manifest.get('cycles', {}).items()):
        cycles.append((pd.Timestamp(info['retrain_date']), key, info.get('val_loss', float('nan'))))
print(f'{len(cycles)} cycles loaded')
for rd, key, vl in cycles:
    print(f'  {key}  {rd.date()}  val_loss={vl:.6f}')

12 cycles loaded
  cycle_000  2015-01-02  val_loss=0.001994
  cycle_001  2015-12-28  val_loss=0.003740
  cycle_002  2016-12-20  val_loss=0.003427
  cycle_003  2017-12-13  val_loss=0.000911
  cycle_004  2018-12-10  val_loss=0.001507
  cycle_005  2019-12-05  val_loss=0.001742
  cycle_006  2020-12-01  val_loss=0.008114
  cycle_007  2021-11-24  val_loss=0.002620
  cycle_008  2022-11-15  val_loss=0.002768
  cycle_009  2023-11-08  val_loss=0.002448
  cycle_010  2024-11-04  val_loss=0.001291
  cycle_011  2025-10-30  val_loss=0.001859


In [7]:
# ── Interactive chart — v2 style: full price history + 10-day forecast overlay ──
# Ratios from pred_cache are converted to CACT price levels: price = anchor × ratio
# Saves HTML to weights/ folder as well as showing inline.

PRED_STEP       = 5
ENTRY_THRESHOLD_PCT = (ENTRY_THRESHOLD - 1.0) * 100.0
HTML_OUT        = os.path.join(WEIGHTS_DIR, 'tkan_v4_prediction_chart.html')

# ── helpers ──────────────────────────────────────────────────────────────────
def _cycle_for_date(t):
    active = None
    for rd, key, vloss in cycles:
        if rd <= t:
            active = (rd, key, vloss)
        else:
            break
    return active

def _build_step(i):
    """Build x/y/hover for predicted and actual 10-day price paths."""
    t   = all_dates[i]
    c0  = float(close.iloc[i])                    # anchor price (today's close)
    ratios = pred_df.iloc[i].values                # d1..d10 ratios
    # predicted prices: anchor × ratio
    pred_x = [t]
    pred_y = [c0]
    act_x  = [t]
    act_y  = [c0]
    hover  = [f"<b>Day 0</b> {t.strftime('%Y-%m-%d')} (anchor)<br>CACT: {c0:,.1f}"]
    errors = []
    for k in range(n_days):
        fi = i + k + 1
        if fi >= len(all_dates):
            break
        fd    = all_dates[fi]
        p_eur = c0 * ratios[k]
        a_eur = float(close.iloc[fi])
        err   = p_eur - a_eur
        errors.append(abs(err))
        pred_x.append(fd); pred_y.append(p_eur)
        act_x.append(fd);  act_y.append(a_eur)
        hover.append(
            f"<b>t+{k+1}</b>  {fd.strftime('%Y-%m-%d')}<br>"
            f"Pred: {p_eur:,.1f}  ({(ratios[k]-1)*100:+.2f}%)<br>"
            f"Actual: {a_eur:,.1f}<br>"
            f"Err: {err:+.1f}")
    max_ratio    = float(pred_df.iloc[i].max())
    signal_on    = max_ratio >= ENTRY_THRESHOLD
    mae          = float(np.mean(errors)) if errors else 0.0
    ci           = _cycle_for_date(t)
    return dict(t=t, c0=c0, pred_x=pred_x, pred_y=pred_y,
                act_x=act_x, act_y=act_y, hover=hover,
                max_ratio=max_ratio, signal_on=signal_on, mae=mae,
                cycle_label=ci[1] if ci else '?',
                val_loss=ci[2] if ci else float('nan'))

def _title(d):
    sig = '▲ ENTRY SIGNAL' if d['signal_on'] else '— no entry'
    return (f"TKAN v4  |  {d['t'].strftime('%Y-%m-%d')}  anchor={d['c0']:,.1f}  "
            f"|  {d['cycle_label']}  val_loss={d['val_loss']:.6f}  "
            f"|  max pred: {(d['max_ratio']-1)*100:+.2f}%  MAE={d['mae']:.1f}  |  {sig}")

# slider index set
slider_idx = list(range(0, len(all_dates), PRED_STEP))
if slider_idx[-1] != len(all_dates) - 1:
    slider_idx.append(len(all_dates) - 1)

# ── build figure ─────────────────────────────────────────────────────────────
fig = go.Figure()

# Trace 0: full CACT history
fig.add_trace(go.Scatter(
    x=all_dates, y=close.values, mode='lines', name='CACT Total Return',
    line=dict(color='royalblue', width=1.5)))

# Trace 1: retrain markers on price line
retrain_x = [rd for rd, k, v in cycles if rd in close.index]
retrain_y = [float(close.loc[rd]) for rd in retrain_x]
fig.add_trace(go.Scatter(
    x=retrain_x, y=retrain_y, mode='markers', name='Retrain',
    marker=dict(size=12, color='orange', symbol='x-thin-open', line=dict(width=3)),
    text=[f"RETRAIN: {rd.strftime('%Y-%m-%d')}" for rd in retrain_x],
    hoverinfo='text'))

# Retrain vertical lines
for rd in retrain_x:
    fig.add_vline(x=rd.timestamp()*1000, line=dict(color='orange', width=1, dash='dot'), opacity=0.4)

# Trace 2: actual 10-day path (restyled by slider)
first = _build_step(slider_idx[0])
fig.add_trace(go.Scatter(
    x=first['act_x'], y=first['act_y'], mode='lines+markers', name='Actual (10d)',
    line=dict(color='royalblue', width=2.5, dash='dash'),
    marker=dict(size=5, color='royalblue')))

# Trace 3: predicted 10-day path (restyled by slider)
sc0 = '#00e676' if first['signal_on'] else '#ff1744'
fig.add_trace(go.Scatter(
    x=first['pred_x'], y=first['pred_y'], mode='lines+markers', name='Predicted (10d)',
    line=dict(color=sc0, width=2.5, dash='dash'),
    marker=dict(size=8, symbol='diamond', color=sc0),
    text=first['hover'], hoverinfo='text'))

# Trace 4: anchor star
fig.add_trace(go.Scatter(
    x=[first['t']], y=[first['c0']], mode='markers', name='Forecast origin',
    marker=dict(size=14, color='lime', symbol='star'),
    text=[first['hover'][0]], hoverinfo='text'))

# ── slider ────────────────────────────────────────────────────────────────────
steps = []
for i in slider_idx:
    d  = _build_step(i)
    sc = '#00e676' if d['signal_on'] else '#ff1744'
    steps.append(dict(
        method='update',
        label=d['t'].strftime('%Y-%m-%d'),
        args=[
            # arrays must have exactly 3 elements — one per trace index [2, 3, 4]
            {'x': [d['act_x'], d['pred_x'], [d['t']]],
             'y': [d['act_y'], d['pred_y'], [d['c0']]],
             'text': [None, d['hover'], [d['hover'][0]]],
             'line.color': ['royalblue', sc, None],
             'marker.color': ['royalblue', sc, 'lime']},
            {'title.text': _title(d)},
            [2, 3, 4],
        ]))

fig.update_layout(
    title=_title(first),
    height=750, width=1500,
    template='plotly_dark',
    hovermode='x unified',
    yaxis_title='CACT Total Return Level',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(b=140),
    annotations=[dict(
        text='green = entry signal  |  red = no entry  |  orange X = model retrain',
        xref='paper', yref='paper', x=0.5, y=-0.12,
        showarrow=False, font=dict(size=12, color='gray'), align='center')],
    sliders=[dict(
        active=0,
        currentvalue=dict(prefix='Forecast from: ', font=dict(size=13), visible=True, xanchor='center'),
        pad=dict(t=55, b=10),
        steps=steps, len=0.95, x=0.025)])

fig.write_html(HTML_OUT)
print(f'HTML saved → {HTML_OUT}')
fig.show()

HTML saved → c:\Personal\Business & Investments\Python codes\btest\research\Index Directional\signals\tkan\versions\v4\weights\tkan_v4_prediction_chart.html


## Signal Construction & Forward-Return Validation

Binary signal: **1 (Long)** when `max(d1..d10) ≥ threshold`, else **0 (Flat)**.

We evaluate both thresholds:
- base: `ENTRY_THRESHOLD` (currently 1.015)
- strict: `1.02` (**max > 2%**)


In [8]:
# ── Signal validation for two criteria: >=1.5% and >2.0% ───────────────────
fwd_ret_10 = close.pct_change(10).shift(-10)
max_ratio  = pred_df.max(axis=1)

thr_map = {
    f'>={(ENTRY_THRESHOLD-1)*100:.1f}%': ENTRY_THRESHOLD,
    '>2.0%': 1.02,
}

rows = []
for label, thr in thr_map.items():
    sig = (max_ratio >= thr).astype(int)
    df_ = pd.DataFrame({'signal': sig, 'fwd_ret_10': fwd_ret_10}).dropna()
    on  = df_.loc[df_['signal'] == 1, 'fwd_ret_10']
    rows.append({
        'criteria': label,
        'signal_pct': on.size / len(df_),
        'mean_10d_fwd': on.mean(),
        'hit_rate': (on > 0).mean(),
        'n_on': int(on.size),
    })

summary = pd.DataFrame(rows).set_index('criteria')
for c in ['signal_pct', 'mean_10d_fwd', 'hit_rate']:
    summary[c] = (summary[c] * 100).round(2)

print('Signal criteria summary (ON days only):')
print(summary[['n_on', 'signal_pct', 'mean_10d_fwd', 'hit_rate']]
      .rename(columns={'signal_pct': 'signal_%', 'mean_10d_fwd': 'mean_10d_fwd_%', 'hit_rate': 'hit_rate_%'})
      .to_string())

# Keep base-threshold signal as default for downstream cells
signal = (max_ratio >= ENTRY_THRESHOLD).astype(int)


Signal criteria summary (ON days only):
          n_on  signal_%  mean_10d_fwd_%  hit_rate_%
criteria                                            
>=1.5%    1685     58.85            0.41       60.18
>2.0%     1473     51.45            0.36       59.95


In [9]:
# ── Equity curve: Long on signal=1, flat on signal=0 ─────────────────────────
daily_ret = close.pct_change().reindex(df_sig.index)

# Strategy: enter at open next day, hold until signal flips or after 10 days
# Simple version: hold position proportional to signal (1 = invested, 0 = flat)
strat_ret  = daily_ret * signal.shift(1).fillna(0)   # lag 1 day (trade next open)
bnh_ret    = daily_ret

strat_equity = (1 + strat_ret).cumprod()
bnh_equity   = (1 + bnh_ret).cumprod()

# ── Per-cycle stats ───────────────────────────────────────────────────────────
print("Per-cycle signal stats:")
print(f"{'Cycle':12s}  {'Retrain':12s}  {'Signal%':>8}  {'MeanFwd':>8}  {'HitRate':>8}")
print("-" * 60)
for rd, key, vl in cycles:
    # OOS period: from retrain date to next retrain (or end)
    next_rds = [r for r, k, v in cycles if r > rd]
    end_rd   = next_rds[0] if next_rds else df_sig.index[-1]
    mask     = (df_sig.index >= rd) & (df_sig.index < end_rd)
    sub      = df_sig[mask]
    if len(sub) == 0:
        continue
    sig_pct  = sub['signal'].mean()
    on_mask  = sub['signal'] == 1
    mfwd     = sub.loc[on_mask, 'fwd_ret_10'].mean() * 100 if on_mask.any() else float('nan')
    hr       = (sub.loc[on_mask, 'fwd_ret_10'] > 0).mean() if on_mask.any() else float('nan')
    print(f"{key:12s}  {rd.date()!s:12s}  {sig_pct:>8.1%}  {mfwd:>8.2f}%  {hr:>8.1%}")

# ── Summary equity chart ──────────────────────────────────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=strat_equity.index, y=strat_equity.values,
                          mode='lines', name='TKAN Signal (Long/Flat)',
                          line=dict(color='#00e676', width=2)))
fig2.add_trace(go.Scatter(x=bnh_equity.index, y=bnh_equity.values,
                          mode='lines', name='Buy & Hold CACT',
                          line=dict(color='royalblue', width=1.5, dash='dot')))

# shade signal-on periods
in_signal = False
sig_start = None
for dt, s in signal.items():
    if s == 1 and not in_signal:
        sig_start = dt; in_signal = True
    elif s == 0 and in_signal:
        fig2.add_vrect(x0=sig_start, x1=dt, fillcolor='rgba(0,230,118,0.07)',
                       line_width=0)
        in_signal = False
if in_signal:
    fig2.add_vrect(x0=sig_start, x1=signal.index[-1],
                   fillcolor='rgba(0,230,118,0.07)', line_width=0)

strat_total = strat_equity.iloc[-1] - 1
bnh_total   = bnh_equity.iloc[-1]   - 1
strat_sharpe = strat_ret.mean() / strat_ret.std() * (252 ** 0.5)
bnh_sharpe   = bnh_ret.mean()   / bnh_ret.std()   * (252 ** 0.5)

fig2.update_layout(
    title=(f"TKAN v4 Signal Equity  |  "
           f"Strategy: {strat_total:.1%}  Sharpe={strat_sharpe:.2f}  |  "
           f"B&H: {bnh_total:.1%}  Sharpe={bnh_sharpe:.2f}  |  "
           f"threshold={ENTRY_THRESHOLD}  signal_freq={signal.mean():.1%}"),
    height=550, template='plotly_dark',
    yaxis_title='Cumulative Return (starting 1.0)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified')
fig2.show()


NameError: name 'df_sig' is not defined

In [ ]:
# ── State machine: dual-exit strategy ────────────────────────────────────────
#
#  FLAT:
#    - signal=1 on day T  →  enter LONG at day T+1 open (today's close as proxy)
#    - signal=0           →  stay flat, no entry
#
#  LONG (holding position from entry_price):
#    EXIT when EITHER condition fires on day T, execute at day T+1 open:
#      (A) TARGET HIT:   close[T] >= entry_price × ENTRY_THRESHOLD  ← take profit
#      (B) MOMENTUM STOP: signal[T] = 0                              ← stop loss
#    whichever comes first. Both can fire on the same day (target hit takes label).
#
#  Execution lag: 1 day throughout (no look-ahead).

signal_full = (pred_df.max(axis=1) >= ENTRY_THRESHOLD).astype(int).reindex(close.index, fill_value=0)

dates_arr  = close.index.tolist()
close_arr  = close.values
sig_arr    = signal_full.values

position   = pd.Series(0, index=close.index, dtype=int)
trade_log  = []
state      = 'FLAT'
entry_idx  = None
entry_price  = None
target_price = None

for i in range(1, len(dates_arr)):
    prev_sig = int(sig_arr[i - 1])     # signal known end-of-day i-1
    c        = float(close_arr[i])     # today's close (execution price proxy)
    t        = dates_arr[i]

    if state == 'FLAT':
        if prev_sig == 1:              # momentum on → enter
            state        = 'LONG'
            entry_idx    = i
            entry_price  = c
            target_price = c * ENTRY_THRESHOLD
            position.iloc[i] = 1

    else:  # LONG
        position.iloc[i] = 1

        target_hit    = c >= target_price          # (A) price reached goal
        momentum_stop = prev_sig == 0              # (B) model no longer bullish

        if target_hit or momentum_stop:
            trade_log.append({
                'entry_date':    dates_arr[entry_idx],
                'exit_date':     t,
                'entry_price':   entry_price,
                'exit_price':    c,
                'target_price':  target_price,
                'hold_days':     i - entry_idx,
                'return':        c / entry_price - 1,
                'exit_reason':   'target' if target_hit else 'momentum_stop',
                'win':           c > entry_price,
            })
            state = 'FLAT'

trades = pd.DataFrame(trade_log)

# ── Summary stats ─────────────────────────────────────────────────────────────
n_target = (trades['exit_reason'] == 'target').sum()
n_stop   = (trades['exit_reason'] == 'momentum_stop').sum()

print(f"Trades:          {len(trades)}")
print(f"  Target exits:  {n_target}  ({n_target/len(trades):.1%})")
print(f"  Momentum stops:{n_stop}   ({n_stop/len(trades):.1%})")
print(f"Win rate:        {trades['win'].mean():.1%}")
print(f"Avg return:      {trades['return'].mean()*100:.2f}%")
print(f"Median ret:      {trades['return'].median()*100:.2f}%")
print(f"Best / Worst:    {trades['return'].max()*100:.2f}%  /  {trades['return'].min()*100:.2f}%")
print(f"Avg hold:        {trades['hold_days'].mean():.1f} days")
print(f"Max hold:        {trades['hold_days'].max()} days")
print(f"Invested:        {position.mean():.1%} of time")
print()
for reason in ['target', 'momentum_stop']:
    sub = trades[trades['exit_reason'] == reason]
    if len(sub):
        print(f"  {reason:16s}  n={len(sub):3d}  "
              f"avg={sub['return'].mean()*100:+.2f}%  "
              f"win={sub['win'].mean():.1%}  "
              f"avg_hold={sub['hold_days'].mean():.1f}d")

# ── Equity curve ──────────────────────────────────────────────────────────────
daily_ret    = close.pct_change()
strat_ret    = daily_ret * position.shift(1).fillna(0)
strat_equity = (1 + strat_ret).cumprod()
bnh_equity   = (1 + daily_ret).cumprod()

strat_total  = strat_equity.iloc[-1] - 1
bnh_total    = bnh_equity.iloc[-1]   - 1
strat_sharpe = strat_ret.mean() / strat_ret.std() * (252 ** 0.5)
bnh_sharpe   = daily_ret.mean() / daily_ret.std() * (252 ** 0.5)

roll_max = strat_equity.cummax()
max_dd   = ((strat_equity - roll_max) / roll_max).min()

print(f"\nEquity summary:")
print(f"  Strategy:   {strat_total:.1%}  Sharpe={strat_sharpe:.2f}  MaxDD={max_dd:.1%}")
print(f"  Buy & Hold: {bnh_total:.1%}  Sharpe={bnh_sharpe:.2f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig3 = go.Figure()

fig3.add_trace(go.Scatter(x=strat_equity.index, y=strat_equity.values,
                          mode='lines', name='TKAN Dual-Exit Strategy',
                          line=dict(color='#00e676', width=2)))
fig3.add_trace(go.Scatter(x=bnh_equity.index, y=bnh_equity.values,
                          mode='lines', name='Buy & Hold CACT',
                          line=dict(color='royalblue', width=1.5, dash='dot')))

# shade invested bands
for _, tr in trades.iterrows():
    clr = 'rgba(0,230,118,0.08)' if tr['exit_reason'] == 'target' else 'rgba(255,23,68,0.06)'
    fig3.add_vrect(x0=tr['entry_date'], x1=tr['exit_date'], fillcolor=clr, line_width=0)

# entry markers
entry_y = strat_equity.reindex(trades['entry_date'].values, method='nearest').values
fig3.add_trace(go.Scatter(
    x=trades['entry_date'].values, y=entry_y, mode='markers', name='Entry',
    marker=dict(symbol='triangle-up', size=10, color='#00e676'),
    text=[f"Entry {r['entry_date'].date()}  @{r['entry_price']:.1f}  target={r['target_price']:.1f}"
          for _, r in trades.iterrows()],
    hoverinfo='text'))

# exit markers — colour by reason
for reason, sym, col in [('target', 'star', '#ffd600'), ('momentum_stop', 'triangle-down', '#ff1744')]:
    sub = trades[trades['exit_reason'] == reason]
    if len(sub) == 0: continue
    eq_y = strat_equity.reindex(sub['exit_date'].values, method='nearest').values
    fig3.add_trace(go.Scatter(
        x=sub['exit_date'].values, y=eq_y, mode='markers',
        name=f"Exit — {reason.replace('_', ' ')}",
        marker=dict(symbol=sym, size=11, color=col),
        text=[f"{reason}  {r['exit_date'].date()}  ret={r['return']*100:+.2f}%  hold={r['hold_days']}d"
              for _, r in sub.iterrows()],
        hoverinfo='text'))

fig3.update_layout(
    title=(f"TKAN v4 — Dual-Exit (Target ★ | Momentum Stop ▼)  |  "
           f"Total: {strat_total:.1%}  Sharpe={strat_sharpe:.2f}  MaxDD={max_dd:.1%}  |  "
           f"B&H: {bnh_total:.1%}  Sharpe={bnh_sharpe:.2f}  |  "
           f"threshold={ENTRY_THRESHOLD}  invested={position.mean():.1%}"),
    height=620, template='plotly_dark',
    yaxis_title='Cumulative Return (starting 1.0)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified')
fig3.show()


Trades:          320
  Target exits:  111  (34.7%)
  Momentum stops:209   (65.3%)
Win rate:        64.1%
Avg return:      0.23%
Median ret:      0.64%
Best / Worst:    7.99%  /  -16.52%
Avg hold:        4.9 days
Max hold:        52 days
Invested:        66.6% of time

  target            n=111  avg=+2.26%  win=100.0%  avg_hold=4.7d
  momentum_stop     n=209  avg=-0.84%  win=45.0%  avg_hold=5.0d

Equity summary:
  Strategy:   101.6%  Sharpe=0.47  MaxDD=-39.4%
  Buy & Hold: 155.2%  Sharpe=0.54
